### Check for the list of genes if we have a overlab 
- variants tested 
- elements tested
- 17.07: email to exceter: 
  - I want to sent the list of the overlapping genes
  - We 

In [1]:
import pandas as pd
import numpy as np
import os
import yaml
import re


# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [2]:
# helpful functions

def get_gene_from_header(header):
    """
    Returns metioned gene from header:
    cardiac_neuro_cava_random:SKI|ENSG00000157933.11|EH38E2778476_fwd_tile1-1 => SKI
    cardiac_neuro_cava_random:ALT_MTOR|ENSG00000198793.14|EH38E1318606_rev_tile1-1_MTOR|ENSG00000198793.14|EH38E1318606|1-11073630-C-T => MTOR
    """
    if not 'cardiac_neuro_cava_random' in header:
        return "NA"
    if ":REF_" in header:
        gene = header.split(":REF_")[1].split("|")[0]
    elif ":ALT_" in header:
        gene = header.split(":ALT_")[1].split("|")[0]
    else:
        gene = header.split(":")[1].split("|")[0]
    return gene

#### Read the Protein list from exceter mail and the metadata file from MPRA
- Get the gene / protein names 
- Find the intersect (lower case)

In [29]:
# read metadata file (only tested)
meta_data_file = pd.read_csv(config['files']['creating']['metadata_table_local_tested_juli'], sep="\t")
meta_data_file = pd.read_csv(config['files']['creating']['region_metadata_table_local_tested_juli'], sep="\t")
meta_data_file
# change start and end to int
meta_data_file['start'] = meta_data_file['start'].astype(int)
meta_data_file['end'] = meta_data_file['end'].astype(int)
# read their protein list
protein_list = pd.read_csv(config['files']['collaborations']['exceter_protein_list'], sep="\t", header=None)
protein_list.columns = ["protein"]
protein_list
# number of different proteins
protein_list['protein'].nunique() # 2923
# check for tested sequences only:
# get all genes from the tested sequence header
meta_data_file['tmp_gene_names'] = meta_data_file['header'].apply(get_gene_from_header)
# make genes lower case
meta_data_file['tmp_gene_names_lower'] = meta_data_file['tmp_gene_names'].str.lower()
meta_data_file
# number of uniqe genes:
meta_data_file['tmp_gene_names_lower'].nunique() # 1049

# why do we have 1049 different genes? Assumed 524 genes


/tmp/ipykernel_38050/1798260235.py:3: DtypeWarning: Columns (8,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  meta_data_file = pd.read_csv(config['files']['creating']['region_metadata_table_local_tested_juli'], sep="\t")


525

#### Investigate general intersect between MPRA and Exceter

In [30]:
# all genes
# set approach:
exceter_protein_set = set(protein_list['protein'].to_list())

mpra_gene_set = set(meta_data_file['tmp_gene_names_lower'].to_list())

# sanity check
len(mpra_gene_set)
len(exceter_protein_set)

# get intersect of both lists:
intersect = exceter_protein_set.intersection(mpra_gene_set)
len(intersect) # 95

print(f"Number of intersecting genes: {len(intersect)}")


# variant specific
# element specific


Number of intersecting genes: 95


#### Investigate the number of elements tested with these genes and the elements measured in the MPRA
- first designed
- second measured

In [31]:
elements_tested_mpra = meta_data_file[(meta_data_file['category'] == 'element') | (meta_data_file['header'].str.contains("cardiac_neuro_cava_random:REF_"))]
elements_tested_mpra.loc[elements_tested_mpra['header'].str.contains('REF_SFTPA1')]
# izumo1

print(f"Number of elements in the MRPA design: {elements_tested_mpra.shape[0]}")
elements_gene_set = set(elements_tested_mpra['tmp_gene_names_lower'].to_list())
print(f"Number of different genes in elements: {len(elements_gene_set)}")

element_intersect = exceter_protein_set.intersection(elements_gene_set)
print(f"Number of intersecting genes between exceter and disease-associated open-chromating MPRA elements: {len(element_intersect)}") # 92

Number of elements in the MRPA design: 27472
Number of different genes in elements: 525
Number of intersecting genes between exceter and disease-associated open-chromating MPRA elements: 95


##### How many elements did we test in the MPRA

##### Read the assignment file and the replicate files  
1. filter based on the number of observed barcodes, 
2. build the overlap 
3. get these sequences from the metadata file (add exceter_overlap column (True if overlap and false if not)) 
4. Only look at this table and get number of tested (not allele alt) (element) / and overlap with variant header (Report these numbers together with the metadata file)

In [32]:
def filter_barcodes(df, print_text, threshold=5, barcode_column="n_obs_bc", verbose=True):
    """
    Filters the given df based on the barcode column, prints the number of unique names in df and returns the filtered df
    """
    df_filtered = df.loc[df[barcode_column] >= threshold]
    if verbose:
        print(f"{name}: {df_filtered['name'].nunique()}")
    return df_filtered

In [33]:
# # load assignemnt file
# assignemnt_file = pd.read_csv(config['files']['final_design']['assignment_file'], sep="\t", header=None)
# assignemnt_file.columns = ['barcode', 'name', 'alignment_info', 'assignment_number']

# # load replicate files

# # example workflow for one replicate
# replicate_1 = pd.read_csv(config['files']['final_design']['barcodes_rep1'], sep="\t", header=None)
# replicate_1.columns = ['barcode', 'dna_counts', 'rna_counts']
# replicate_1['replicate'] = 1

# print(f"Number of barcodes in replicate 1: {replicate_1['barcode'].nunique()}")
# # print(f"Number of barcodes in replicate 1: {replicate_1.shape[0]}")

# # same for replicate 2
# replicate_2 = pd.read_csv(config['files']['final_design']['barcodes_rep2'], sep="\t", header=None)
# replicate_2.columns = ['barcode', 'dna_counts', 'rna_counts']
# replicate_2['replicate'] = 2

# print(f"Number of barcodes in replicate 2: {replicate_2['barcode'].nunique()}")
# # print(f"Number of barcodes in replicate 2: {replicate_2.shape[0]}")

# # and 3
# replicate_3 = pd.read_csv(config['files']['final_design']['barcodes_rep3'], sep="\t", header=None)
# replicate_3.columns = ['barcode', 'dna_counts', 'rna_counts']
# replicate_3['replicate'] = 3

# print(f"Number of barcodes in replicate 3: {replicate_3['barcode'].nunique()}")
# # print(f"Number of barcodes in replicate 3: {replicate_3.shape[0]}")

# # Number of barcodes in replicate 1: 4695996
# # Number of barcodes in replicate 2: 4704637
# # Number of barcodes in replicate 3: 4734536

# # merge with assignment file
# replicate_1_assignment = pd.merge(replicate_1, assignemnt_file[['barcode', 'name', 'assignment_number']], on='barcode', how='left')

# replicate_2_assignment = pd.merge(replicate_2, assignemnt_file[['barcode', 'name', 'assignment_number']], on='barcode', how='left')

# replicate_3_assignment = pd.merge(replicate_3, assignemnt_file[['barcode', 'name', 'assignment_number']], on='barcode', how='left')

In [34]:
# # add number of observed barcodes
# replicate_1_assignment['n_obs_bc'] = replicate_1_assignment.groupby('name')['barcode'].transform('count')
# replicate_2_assignment['n_obs_bc'] = replicate_2_assignment.groupby('name')['barcode'].transform('count')
# replicate_3_assignment['n_obs_bc'] = replicate_3_assignment.groupby('name')['barcode'].transform('count')

In [35]:
# # filter based on barcode number and get overlap
# min_barcode_threshold = config['analysis_info']['min_barcode_threshold']
# _total_number_oligos = config["analysis_info"]["total_number_oligos"]

# rep_df_dict = {
#     'Rep1': replicate_1_assignment,
#     'Rep2': replicate_2_assignment,
#     'Rep3': replicate_3_assignment
# }

# replicate_oligo_sets = []
# print(f"Minimum number of barcodes: {min_barcode_threshold}")
# for name, df in rep_df_dict.items():
#     rep = filter_barcodes(df, print_text=name, threshold=min_barcode_threshold)
#     # compute set of oligos per replicate
#     replicate_oligo_sets.append(set(rep['name'].to_list()))
# # compute the overlap between replicates
# replicate_overlap = replicate_oligo_sets[0].intersection(replicate_oligo_sets[1]).intersection(replicate_oligo_sets[2])
# print(f'{len(replicate_overlap)} oligos overlap')
# # compute the proportion compared to the overall number of oligos (80215)
# print(f'{round(len(replicate_overlap) / _total_number_oligos, 4)} proportion of oligos overlap')
# # compute the union of oligos between replicates
# # replicate_union = replicate_oligo_sets[0].union(replicate_oligo_sets[1]).union(replicate_oligo_sets[2])
# # # Compute the proportion compared to the maximal possible overlap (union of all replicates without filtering)
# # print(f'{round(len(replicate_overlap) / len(replicate_union), 4)} proportion of oligos overlap in the union')
# # print(f'{len(replicate_union)} oligos in total')
# # # compute the proportion compared to the overall number of oligos (80215)
# # print(f'{round(len(replicate_union) / _total_number_oligos, 4)} proportion of oligos union')

In [36]:
assigned_barcodes = pd.read_csv(config['files']['final_design']['mprasnakeflow_final_resequencing_assigned_barcodes'], sep="\t")
assigned_barcodes

,Barcode,name,dna_count_replicate_1,rna_count_replicate_1,dna_count_replicate_2,rna_count_replicate_2,dna_count_replicate_3,rna_count_replicate_3
0,AAATACAAGCTGTTC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,4.0,12.0,6.0,10.0,3.0,17.0
1,AAGACTCGACAACCA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,4.0,9.0,4.0,10.0,3.0,21.0
2,AATACCCAAGAACCC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,1.0,2.0,1.0,4.0,NaN,NaN
3,AATATGAGGGTTAAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,8.0,18.0,8.0,18.0,7.0,23.0
4,AATGGGGTCCGTAAA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,5.0,2.0,7.0,3.0,8.0
...,...,...,...,...,...,...,...,...
5443407,TTGAGGCAGCTAGAT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,1.0,5.0,NaN,NaN
5443408,TTGCCAAGCGGTACT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,2.0,12.0,6.0,15.0
5443409,GAAATGACTCATATT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,NaN,NaN,2.0,1.0
5443410,GTCTGACGGAAGAAC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,NaN,NaN,1.0,5.0


In [37]:
# drop rows with na values in one the columns: ['dna_count_replicate_1', 'rna_count_replicate_1', 'dna_count_replicate_2', 'rna_count_replicate_2', 'dna_count_replicate_3', 'rna_count_replicate_3']
# Drop columns with any NA values
assigned_barcodes_cleaned = assigned_barcodes.dropna(axis=1)

In [38]:
assigned_barcodes['name'].nunique() # 69297

69297

In [39]:
assigned_barcodes_cleaned['name'].nunique() # 69297

69297

In [40]:
replicate_cleaned_overlapped = set(assigned_barcodes_cleaned['name'].to_list())

In [41]:
meta_data_file.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_gene_names', 'tmp_gene_names_lower'],
      dtype='object')

In [42]:
# list comprehension to filter the replicate_overlap for names which start with cardiac_neuro_cava_random
tested_replicate_overlap_10bcs = [name for name in replicate_overlap if 'cardiac_neuro_cava_random' in name]
tested_replicate_cleaned_overlap_10bcs = [name for name in replicate_cleaned_overlapped if 'cardiac_neuro_cava_random' in name]
len(tested_replicate_overlap_10bcs)

# read their protein list => exceter_protein_set
protein_list = pd.read_csv(config['files']['collaborations']['exceter_protein_list'], sep="\t", header=None)
protein_list.columns = ["protein"]
exceter_protein_set = set(protein_list['protein'].to_list())

def in_count_measurements(header):
    """
    Returns True if the header is in the measurements of our filtered MPRA results
    """
    return header in tested_replicate_cleaned_overlap_10bcs

def in_count_measurements_old(header):
    """
    Returns True if the header is in the measurements of our filtered MPRA results
    """
    return header in tested_replicate_overlap_10bcs

def in_exceter_gene_list(header):
    """
    Returns true if header is associated to a gene within the exceter gene list (aka protein list)
    """
    gene_name = get_gene_from_header(header)
    gene_name_lower = gene_name.lower()

    return gene_name_lower in exceter_protein_set


In [43]:
# add column to metadata file
meta_data_file = meta_data_file.assign(in_count_measurements=meta_data_file['header'].apply(in_count_measurements))

# add exceter_overlap column
meta_data_file = meta_data_file.assign(in_exceter_gene_list=meta_data_file['header'].apply(in_exceter_gene_list))

# # find overlap with genes of interest from exceter
# meta_data_measured_sequences = meta_data_file.loc[meta_data_file['in_count_measurements']]

# filter for this new column and get the count based on element or variant
measurements_exceter_gene_list = meta_data_file[(meta_data_file['in_count_measurements']) & (meta_data_file['in_exceter_gene_list'])]


In [44]:
measurements_exceter_gene_list.columns

Index(['header', 'sequence', 'tmp_label', 'name', 'category', 'class',
       'source', 'ref', 'variant_class', 'variant_pos', 'SPDI', 'allele',
       'info', 'tmp_matching_header', 'chr', 'start', 'end', 'region_name',
       'strand', 'tmp_gene_names', 'tmp_gene_names_lower',
       'in_count_measurements', 'in_exceter_gene_list'],
      dtype='object')

In [45]:
meta_data_file.shape[0] # 73846 (=> only tested and without the not matchable REF and ALT sequences)

73846

In [46]:
meta_data_file.head()

,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,...,tmp_matching_header,chr,start,end,region_name,strand,tmp_gene_names,tmp_gene_names_lower,in_count_measurements,in_exceter_gene_list
0,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGTCCCAGCTCCCCACTGATGTGAAAGGTGGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,SNP,83.0,...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,chr1,2179507,2179777,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,SKI,ski,True,False
1,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTGATCTGCCCTGTCCGTGACGCTTCTGCT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,SNP,181.0,...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,chr1,2191262,2191532,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,SKI,ski,False,False
2,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCTCTGGGTGACCCGGAGAACACCAAGGCTG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,SNP,43.0,...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,chr1,2191971,2192241,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,SKI,ski,False,False
3,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTCCATGCGGTGGCCACAGCCTCGGGTGAGTTC...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,SNP,116.0,...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,chr1,2192249,2192519,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,SKI,ski,True,False
4,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,AGGACCGGATCAACTGGACTCCGGTGCCTTCGCATTCCCGAGCTGT...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,SNP,205.0,...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,chr1,2192936,2193206,cardiac_neuro_cava_random:SKI|ENSG00000157933....,+,SKI,ski,True,False


In [47]:
# Report number of overlapping genes:
print(f"Number of overlapping genes in our measurments: {measurements_exceter_gene_list['tmp_gene_names_lower'].nunique()}")
# Report number of variants with overlapping genes:
print(f"Number of overlapping genes within the set of designed variants: {measurements_exceter_gene_list[measurements_exceter_gene_list['allele'] == 'alt']['tmp_gene_names_lower'].nunique()}") # 3994
print(f"Number of variants associated to the exceter gene list: {measurements_exceter_gene_list[measurements_exceter_gene_list['allele'] == 'alt'].shape[0]}") # 7320
# Report number of elements with overlapping genes:
print(f"Number of overlapping genes within the set of designed elements: {measurements_exceter_gene_list[measurements_exceter_gene_list['allele'] != 'alt']['tmp_gene_names_lower'].nunique()} ")
print(f"Number of elements associated to the exceter gene list: {measurements_exceter_gene_list[measurements_exceter_gene_list['allele'] != 'alt'].shape[0]}")
measurements_exceter_gene_list # 11179

# number of variants with association to gene list of exceter
variants_measurments_in_exceter_list = measurements_exceter_gene_list[measurements_exceter_gene_list['category'] == 'variant'].shape[0] # 9931

# number of elements with association to gene_list of exceter
measurements_exceter_gene_list[~(measurements_exceter_gene_list['allele'] == 'alt')].shape[0] # 3994 # 4073

Number of overlapping genes in our measurments: 95
Number of overlapping genes within the set of designed variants: 94
Number of variants associated to the exceter gene list: 7320
Number of overlapping genes within the set of designed elements: 95 
Number of elements associated to the exceter gene list: 4073


4073

In [49]:
# difference between variant and element set
variant_associated_gene_overlap = set(measurements_exceter_gene_list[measurements_exceter_gene_list['allele'] == 'alt']['tmp_gene_names_lower'].to_list())
element_associated_gene_overlap = set(measurements_exceter_gene_list[measurements_exceter_gene_list['allele'] != 'alt']['tmp_gene_names_lower'].to_list())

len(element_associated_gene_overlap)
element_associated_gene_overlap - variant_associated_gene_overlap

{'rras'}

In [58]:
# store the overlapping genes
overlap_exceter_mpra = pd.DataFrame({'overlap_exceter_NGN2_80k_MPRA': list(element_associated_gene_overlap)})
# add columns: measured_elements: and measured_variants:
overlap_exceter_mpra['measured_elements'] = overlap_exceter_mpra['overlap_exceter_NGN2_80k_MPRA'].apply(lambda x: x in element_associated_gene_overlap)
overlap_exceter_mpra['measured_variants'] = overlap_exceter_mpra['overlap_exceter_NGN2_80k_MPRA'].apply(lambda x: x in variant_associated_gene_overlap)

# overlap_exceter_mpra.to_csv(config['files']['collaborations']['exceter_overlapping_genes_output'], sep="\t", index=False)

: 

In [22]:
# # get numbers for the designed variants and elements (for the design):
# meta_data_exceter_gene_list = meta_data_file[(meta_data_file['in_exceter_gene_list'])]
# # Report number of overlapping genes:
# print(f"Number of overlapping genes in our measurments: {meta_data_exceter_gene_list['tmp_gene_names_lower'].nunique()}")
# # Report number of variants with overlapping genes:
# print(f"Number of overlapping genes within the set of designed variants: {meta_data_exceter_gene_list[meta_data_exceter_gene_list['allele'] == 'alt']['tmp_gene_names_lower'].nunique()}") # 3994
# print(f"Number of variants associated to the exceter gene list: {meta_data_exceter_gene_list[meta_data_exceter_gene_list['allele'] == 'alt'].shape[0]}")
# # Report number of elements with overlapping genes:
# print(f"Number of overlapping genes within the set of designed elements: {meta_data_exceter_gene_list[meta_data_exceter_gene_list['allele'] != 'alt']['tmp_gene_names_lower'].nunique()} ")
# print(f"Number of elements associated to the exceter gene list: {meta_data_exceter_gene_list[meta_data_exceter_gene_list['allele'] != 'alt'].shape[0]}")

In [25]:
# write the exceter specific metadata file:

columns_of_interest = ['name', 'sequence', 'category', 'class', 'source', 'ref', 'chr', 'start', 'end', 'strand', 'variant_class', 'variant_pos', 'SPDI', 'allele', 'info']

exceter_metadata_file_final = measurements_exceter_gene_list[columns_of_interest]
exceter_metadata_file_final.to_csv(config['files']['collaborations']['exceter_meta_data_output'], sep="\t", index=None)

In [ ]:
# store list of genes
overlap_genes_list = measurements_exceter_gene_list['tmp_gene_names_lower'].unique()

In [24]:
measurements_exceter_gene_list.loc[measurements_exceter_gene_list['SPDI'] == 'NC_000003.12:9979051:G:A']['name'].to_list()
measurements_exceter_gene_list.loc[measurements_exceter_gene_list['SPDI'] == 'NC_000003.12:9979051:G:A']

,header,sequence,tmp_label,name,category,class,source,ref,variant_class,variant_pos,...,tmp_matching_header,chr,start,end,region_name,strand,tmp_gene_names,tmp_gene_names_lower,in_count_measurements,in_exceter_gene_list
30171,cardiac_neuro_cava_random:ALT_CRELD1|ENSG00000...,AGGACCGGATCAACTGACTCCATCTCAAAAAAACAACAACAACAAA...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_CRELD1|ENSG00000...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,SNP,239.0,...,cardiac_neuro_cava_random:ALT_CRELD1|ENSG00000...,chr3,9978812,9979082,cardiac_neuro_cava_random:CRELD1|ENSG000001637...,+,CRELD1,creld1,True,True
30172,cardiac_neuro_cava_random:ALT_CRELD1|ENSG00000...,AGGACCGGATCAACTAACATCTTCAAGTCTCAGCATCTCCACAAAG...,cardiac_neuro_cava_random,cardiac_neuro_cava_random:ALT_CRELD1|ENSG00000...,variant,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,SNP,27.0,...,cardiac_neuro_cava_random:ALT_CRELD1|ENSG00000...,chr3,9979024,9979294,cardiac_neuro_cava_random:CRELD1|ENSG000001637...,+,CRELD1,creld1,True,True


In [50]:
exceter_metadata_file['SPDI'].value_counts() # 7320

NC_000003.12:9979051:G:A     2
NC_000019.10:13158256:T:C    2
NC_000022.11:41098920:G:A    1
NC_000022.11:41094921:T:C    1
NC_000022.11:41069678:C:T    1
                            ..
NC_000015.10:67106594:G:T    1
NC_000015.10:67104874:C:T    1
NC_000015.10:67104802:A:G    1
NC_000015.10:67104712:C:A    1
NC_000015.10:67109638:G:A    1
Name: SPDI, Length: 7318, dtype: int64

##### Investigating numbers of in_count_measurements + in_exceter_gene_list + element + variant => found REF without matching ALT

In [55]:
meta_data_file[meta_data_file['in_count_measurements']] # 63050
meta_data_file[meta_data_file['in_exceter_gene_list']] # 13340
meta_data_file[(meta_data_file['name'].str.startswith('cardiac_neuro_cava_random')) & ((meta_data_file['category'] == 'element') | meta_data_file['name'].str.contains('REF_'))].shape[0] # 27566

# REF: 18582

27566

In [54]:
18582 + 8994

27576

In [53]:
meta_data_file.shape[0]

73940

In [52]:
meta_data_file[meta_data_file['name'].str.startswith('cardiac_neuro_cava_random')].shape[0]

73940

In [51]:
meta_data_file[meta_data_file['category'] == 'element'] # 8994
meta_data_file[meta_data_file['name'].str.contains('REF_')].shape[0] # 18582

18582

In [57]:
# element + REF in name => REF without ALT
meta_data_file[(meta_data_file['category'] == 'element') & meta_data_file['name'].str.contains('REF_')] # 10
meta_data_file[(meta_data_file['category'] == 'element') & meta_data_file['name'].str.contains('REF_')]['name'].to_list() # 10

['cardiac_neuro_cava_random:REF_FBXO28|ENSG00000143756.12|EH38E2868786_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_MYL2|ENSG00000111245.17|EH38E3040936_rev_tile1-1',
 'cardiac_neuro_cava_random:REF_MEIS2|ENSG00000134138.22|EH38E3127800_rev_tile1-1',
 'cardiac_neuro_cava_random:REF_CHD3|ENSG00000170004.19|EH38E3207402~KDM6B|ENSG00000132510.11|EH38E3207402_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_PIGS|ENSG00000087111.22|EH38E3215655_rev_tile1-1',
 'cardiac_neuro_cava_random:REF_CIC|ENSG00000079432.9|EH38E3307613_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_GIGYF2|ENSG00000204120.16|EH38E3407187_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_WDFY3|ENSG00000163625.17|EH38E2309079_rev_tile1-1',
 'cardiac_neuro_cava_random:REF_GATAD1|ENSG00000157259.8|EH38E3785709_fwd_tile1-1',
 'cardiac_neuro_cava_random:REF_NLGN3|ENSG00000196338.15|EH38E3937045_fwd_tile1-1']

In [22]:
print(len(tested_replicate_overlap_10bcs))
# filter metadata file for sequences in replicate overlap:
tested_metadata_replicate_overlap_10bcs = meta_data_file.loc[meta_data_file['header'].isin(tested_replicate_overlap_10bcs)]
tested_metadata_replicate_overlap_10bcs

63050


63050

#### Investigate measured variant overlap between MPRA and Exceter
- 94 genes

In [ ]:
# variant specific
# read the mpralm results
mpralm_results = pd.read_csv(config['files']['creating']['toptable_bcMPRAlm_final_resequencing'], sep="\t")
tested_mpralm_results = mpralm_results.loc[mpralm_results['variant_id'].str.startswith('cardiac_neuro_cava_random'), :]
tested_mpralm_results

print(f"Number of measured variants in the MRPA design: {tested_mpralm_results.shape[0]}")

# get all genes from the tested sequence header
tested_mpralm_results = tested_mpralm_results.assign(tmp_gene_names=tested_mpralm_results['variant_id'].apply(get_gene_from_header))
# make genes lower case
tested_mpralm_results['tmp_gene_names_lower'] = tested_mpralm_results['tmp_gene_names'].str.lower()
mpralm_gene_set = set(tested_mpralm_results['tmp_gene_names_lower'].to_list())
print(f"Number of different genes in variants: {len(mpralm_gene_set)}")
len(mpralm_gene_set) # 795

variant_intersect = exceter_protein_set.intersection(mpralm_gene_set)
print(f"Number of intersecting genes between exceter and disease-associated open-chromating MPRA variant: {len(variant_intersect)}") # 94

Number of measured variants in the MRPA design: 36414
Number of different genes in variants: 522
Number of intersecting genes between exceter and disease-associated open-chromating MPRA variant: 94


##### How many variants with these genes did we test?

In [ ]:
# get all variants from the mpralm results with these genes (variant_intersect)
variant_intersect_mpralm = tested_mpralm_results[tested_mpralm_results['tmp_gene_names_lower'].isin(variant_intersect)]
variant_intersect_mpralm['variant_id'].nunique() # 6679

# how are these distributed over the genes?

# how are these distributed over the variant types? (common, rare, ultra-rare, singleton?)

# How are these distributed over the different gene sets? (neuro, cardiac, cava, random)


6679

#### Investigating the set difference: 

In [ ]:
# difference in gene list between measured variants and designed elements (regions + references)
mpralm_gene_set - elements_gene_set # set()
elements_gene_set - mpralm_gene_set # {'chrm2', 'rras', 'tsc2'}

{'chrm2', 'rras', 'tsc2'}

In [ ]:
# difference between intersects
variant_intersect - element_intersect # set()
element_intersect - variant_intersect # {'rras'}


{'rras'}